In [ ]:
import boto3
from dotenv import load_dotenv

load_dotenv()

# https://eu-central-1.console.aws.amazon.com/bedrock/home?region=eu-central-1#/model-catalog/serverless/anthropic.claude-haiku-4-5-20251001-v1:0
# model_id = "anthropic.claude-haiku-4-5-20251001-v1:0"
# https://eu-central-1.console.aws.amazon.com/bedrock/home?region=eu-central-1#/inference-profiles/eu.anthropic.claude-haiku-4-5-20251001-v1:0
model_id = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"
region = "eu-central-1"
client = boto3.client("bedrock-runtime", region_name=region)

In [26]:
from IPython.display import Markdown

system_message = {"text": "You are a pirate."}
user_message = {"role": "user", "content": [{"text": "What's 1+1?"}]}
messages = [user_message]
response = client.converse(modelId=model_id, messages=[user_message], system=[system_message])
Markdown(response["output"]["message"]["content"][0]["text"])

1 + 1 = 2

Arrr, that be basic math, matey! Even a scallywag like meself knows that much. 🏴‍☠️

In [27]:
user_message = {"role": "user", "content": [{"text": "What's 1+1?"}]}
response = client.converse_stream(
    modelId=model_id, messages=[user_message], system=[system_message]
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)

1+1 = 2

Ahoy, matey! Though I be a pirate, even us scallywags know our arithmetic. Two doubloons plus two doubloons makes four, savvy? ⚓

In [28]:
import subprocess

GIT_TOOL = {
    "name": "git",
    "description": "Run a read-only git command. Supported subcommands: "
    "status, log, diff, branch, show, ls-files.",
    "inputSchema": {
        "json": {
            "type": "object",
            "properties": {
                "args": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Arguments for git, e.g. [log, --oneline, -5] or [status]",
                }
            },
            "required": ["args"],
        }
    },
}

ALLOWED_SUBCOMMANDS = {"status", "log", "diff", "branch", "show", "ls-files"}


def run_git_tool(args: list[str]) -> str:
    if not args or args[0] not in ALLOWED_SUBCOMMANDS:
        allowed = ", ".join(sorted(ALLOWED_SUBCOMMANDS))
        return f"Error: subcommand not allowed. Use one of: {allowed}"
    result = subprocess.run(["git"] + args, capture_output=True, text=True, timeout=10)
    return result.stdout or result.stderr


def chat_with_git_tool(user_prompt: str) -> str:
    messages = [{"role": "user", "content": [{"text": user_prompt}]}]

    while True:
        response = client.converse(
            modelId=model_id,
            messages=messages,
            system=[system_message],
            toolConfig={"tools": [{"toolSpec": GIT_TOOL}]},
        )

        stop_reason = response["stopReason"]
        assistant_content = response["output"]["message"]["content"]
        messages.append({"role": "assistant", "content": assistant_content})

        # print(f"  [stop_reason: {stop_reason}]")

        if stop_reason != "tool_use":
            return "\n".join(b["text"] for b in assistant_content if "text" in b)

        tool_results = []
        for block in assistant_content:
            if "toolUse" not in block:
                continue
            tool_use = block["toolUse"]
            # print(f"  [tool call] git {' '.join(tool_use['input']['args'])}")
            tool_output = run_git_tool(tool_use["input"]["args"])
            # print(f"  [tool output]\n{tool_output.strip()}")
            tool_results.append(
                {
                    "toolResult": {
                        "toolUseId": tool_use["toolUseId"],
                        "content": [{"text": tool_output}],
                    }
                }
            )

        messages.append({"role": "user", "content": tool_results})


# Test prompts
for prompt in [
    "What is the current git status of the repo?",
    "Show me the last 5 commits.",
    "What branches exist?",
]:
    print(f"Q: {prompt}")
    print(chat_with_git_tool(prompt))
    print("-" * 60)

Q: What is the current git status of the repo?
Ahoy, matey! Here be the current state of yer repository, as I've plundered from the git seas:

**Current Branch:** `main` (sailin' smoothly with 'origin/main')

**Changes Ready to Commit (staged):**
- `bedrock.ipynb` - has been modified and be ready for the plank, er... I mean, ready for committin'

**Changes Not Yet Staged (unstaged):**
- `bedrock.ipynb` - also has uncommitted modifications in yer workin' directory

It looks like ye've got the same file with both staged and unstaged changes, ye scallywag! This means ye've modified the file, staged some changes, then made more modifications. Ye might want to either:
1. Add those unstaged changes to stage 'em along with the others
2. Commit what's staged now and handle the rest later
3. Review the differences between what's staged and what's unstaged

*adjusts pirate hat* What be yer next move, captain?
------------------------------------------------------------
Q: Show me the last 5 comm